# Cronache dell'Immortalità ⏳
**Data Storytelling a cura di Salvatore Rudisi**

---

## 🎯 L'Obiettivo Narrativo
Viviamo nell'era dell'iper-consumismo digitale, un'epoca definita dalla "cultura dello scarto" e da una soglia dell'attenzione sempre più frammentata. Siamo abituati a contenuti progettati per essere consumati in un weekend e immediatamente sostituiti dal prossimo trend algoritmico. Eppure, nel panorama degli anime, assistiamo a un fenomeno controintuitivo: opere che sfidano le leggi del mercato, restando sulla cresta dell'onda per decenni.

Mentre migliaia di titoli affogano nel dimenticatoio dopo appena 12 episodi, alcuni franchise sembrano aver trovato la formula per l'immortalità commerciale. Ma è davvero solo merito di budget infiniti e "inerzia" produttiva, o c'è un'alchimia profonda che lega l'evoluzione della trama alla fedeltà generazionale del pubblico?

In questa analisi indagheremo la longevità come resilienza, cercando di capire cosa permetta a un anime di trasformarsi da semplice prodotto d'intrattenimento a pilastro culturale inamovibile.

**Esploreremo:**
- Capitolo 1: **Storia della produzione**
- Capitolo 2: **I Segreti della Sopravvivenza**
- Capitolo 3: **Il "Club dei Decennali": Qualità o Inerzia?**
- Capitolo 4: **Il cimitero degli spettatori**
- Capitolo 5: **I Titani del Presente**

In [117]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ast
import sys
import os
from pathlib import Path
import plotly.express as px

# Add project root to sys.path
current_dir = Path(os.getcwd())
# Assuming we are in solution/notebooks or root. 
# Depending on where this notebook runs, we need to find 'solution' package.
if 'solution' not in os.listdir(current_dir):
    # Try moving up
    if 'solution' in os.listdir(current_dir.parent):
        sys.path.append(str(current_dir.parent))
    elif 'solution' in os.listdir(current_dir.parent.parent):
        sys.path.append(str(current_dir.parent.parent))

try:
    from solution.src.data import load_processed
    print("✅ Library solution.src.data loaded.")
except ImportError:
    print("⚠️ Could not import solution.src.data. Please ensure you are running from the correct directory.")



✅ Library solution.src.data loaded.


In [118]:
# Configuration
pd.set_option('display.max_columns', None)
px.defaults.template = "plotly_dark"

### 🛠️ Preparazione dei Dati
Per studiare l'evoluzione e la sopravvivenza degli anime, abbiamo bisogno di dati puliti e coerenti.
In questa fase tecnica andiamo a:
1.  **Caricare e Unire** le informazioni anagrafiche (`details`) con le statistiche del pubblico (`stats`).
2.  **Calcolare il Tempo:** Trasformiamo le date grezze in formati utilizzabili per calcolare la durata esatta di ogni show.
3.  **Filtrare il Rumore:** Poiché ci interessano le *serie* e la loro longevità, escludiamo Film, OAV singoli e opere non pertinenti. Vogliamo analizzare solo le storie che si sono sviluppate nel tempo.

In [119]:
# Data Loading
print("Loading data...")
try:
    df_details = load_processed("details")
    df_stats = load_processed("stats")
    print(f"Details loaded: {df_details.shape}")
    print(f"Stats loaded: {df_stats.shape}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback/Debug note
    
# Data Cleaning & Preparation
print("Cleaning data...")

# 1. Parse Dates
df_details['start_date'] = pd.to_datetime(df_details['start_date'], errors='coerce')
df_details['end_date'] = pd.to_datetime(df_details['end_date'], errors='coerce') # Importante: errors='coerce' trasforma errori in NaT

# Extract Start Year
df_details['start_year'] = df_details['start_date'].dt.year

# Calculate Duration (Time) in Days
df_details['duration_days'] = (df_details['end_date'] - df_details['start_date']).dt.days

# 2. Parse Genres and Studios
def safe_eval(x):
    if isinstance(x, list):
        return x
    try:
        if pd.isna(x): return []
        return ast.literal_eval(str(x))
    except:
        return []
df_details['genres_list'] = [safe_eval(x) for x in df_details['genres']]
df_details['studios_list'] = [safe_eval(x) for x in df_details['studios']]

# 3. Handle Episodes
df_details['episodes'] = pd.to_numeric(df_details['episodes'], errors='coerce')

# 4. Merge
cols_to_merge = ['anime_id', 'watching', 'completed', 'dropped', 'plan_to_watch', 'total']
df = df_details.merge(df_stats[cols_to_merge], on='anime_id', how='left')

# 5. Rimuoviamo i tipi non pertinenti
if 'type' in df.columns:
    types_to_exclude = ['Movie', 'Music', 'Special', 'CM', 'PV']
    df = df[~df['type'].isin(types_to_exclude)]
    
# 6. PULIZIA DATE
print(f"Righe prima della pulizia date: {len(df)}")

# Logica: Rimuoviamo chi ha status "finito" (cioè NON Currently Airing) ma data di fine mancante.
# In altre parole, teniamo SOLO chi:
# A) È ancora in corso ('Currently Airing')
# B) Ha una data di fine valida (.notna())
df = df[
    (df['status'] == 'Currently Airing') | (df['end_date'].notna())
]
print(f"Righe dopo la pulizia date: {len(df)}")

print(f"Data cleaning complete.")
df.head(2)

Loading data...
Details loaded: (28955, 29)
Stats loaded: (28939, 27)
Cleaning data...
Righe prima della pulizia date: 17528
Righe dopo la pulizia date: 10706
Data cleaning complete.


,anime_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,end_date,synopsis,rank,popularity,members,favorites,genres,studios,themes,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming,start_year,duration_days,genres_list,studios_list,watching,completed,dropped,plan_to_watch,total
7,299,.hack//Liminality,.hack//LIMINALITY,https://myanimelist.net/anime/299/hack__Limina...,https://cdn.myanimelist.net/images/anime/7/230...,OVA,Finished Airing,6.58,17130,2002-06-20 00:00:00+00:00,2003-04-10 00:00:00+00:00,"While playing the newly released MMORPG ""The W...",6740,4178,34986,35,['Mystery'],['Bee Train'],['Video Game'],[],Original,PG-13 - Teens 13 or older,4,,2002,"['Bandai Visual', 'Bandai', 'CyberConnect2']",[],['Bandai Entertainment'],[],2002.0,294.0,[Mystery],[Bee Train],858.0,24478.0,998.0,7504.0,34987.0
8,9332,.hack//Quantum,.hack//Quantum,https://myanimelist.net/anime/9332/hack__Quantum,https://cdn.myanimelist.net/images/anime/1894/...,OVA,Finished Airing,7.11,20412,2011-01-28 00:00:00+00:00,2011-04-07 00:00:00+00:00,"Tobias, Mary, and Sakuya challenge the impregn...",3960,3808,42217,72,"['Action', 'Adventure', 'Fantasy']",['Kinema Citrus'],['Video Game'],[],Original,PG-13 - Teens 13 or older,3,,2011,"['Bandai Visual', 'flying DOG', 'Showgate', 'B...",[],['Funimation'],['Crunchyroll'],2011.0,69.0,"[Action, Adventure, Fantasy]",[Kinema Citrus],1156.0,28997.0,683.0,10752.0,42223.0


## Capitolo 1: Storia della Produzione 📈
Prima di misurare la *durata* delle vite, contiamo le *nascite*. L'animazione giapponese è passata da prodezza artigianale a industria di massa. 
Osservando il numero di nuove serie lanciate ogni anno, possiamo tracciare le "Golden Age" e l'eventuale saturazione del mercato odierno. Siamo di fronte a una bolla produttiva?



In [146]:
# Grafico 1: Evoluzione Temporale del Numero di Produzioni

production_counts = df['start_year'].value_counts().sort_index().reset_index()
production_counts.columns = ['start_year', 'count']

production_counts = production_counts[
    (production_counts['start_year'] >= 1960) & 
    (production_counts['start_year'] <= 2025)
]

fig1 = px.bar(production_counts, x='start_year', y='count',
              title='1. L\'Esplosione Produttiva - Anime Prodotti per Anno',
              labels={'start_year': 'Anno di Inizio', 'count': 'Numero di Anime'},
              color='count', color_continuous_scale='Plasma',
              template='plotly_dark')

fig1.update_layout(xaxis_title="Anno", yaxis_title="Totale Produzioni", bargap=0.1)
fig1.show()

### La Corsa contro il Tempo ⏳
Ma quanto sopravvivono realmente queste opere?
In questo grafico mettiamo alla prova la **longevità**: ogni punto rappresenta una serie. Sull'asse orizzontale vediamo quando è nata, su quello verticale per quanti anni è rimasta (o sta rimanendo) in onda.
Le "scie verdi" che partono dagli anni '70 o '90 e salgono verso l'alto sono i veri immortali: serie che non hanno mai smesso di essere trasmesse.

In [147]:
# Grafico 2: Scatter Plot - Anno di Inizio vs Durata (Anni)

df_duration = df.copy()

df_duration['calc_end_date'] = pd.to_datetime(df_duration['end_date'].fillna(pd.Timestamp.now(tz='UTC')), utc=True)
df_duration['start_date'] = pd.to_datetime(df_duration['start_date'], errors='coerce', utc=True)

df_duration['end_date_str'] = df_duration['end_date'].apply(lambda x: x.strftime('%Y-%m-%d') if pd.notnull(x) else 'Currently Airing')

df_duration['years_running'] = (df_duration['calc_end_date'] - df_duration['start_date']).dt.days / 365.25

df_duration = df_duration[
    (df_duration['status'] != 'Not yet aired') & 
    (df_duration['years_running'] > 0) & 
    (df_duration['start_year'] >= 1960) & 
    (df_duration['start_year'] <= pd.Timestamp.now().year)
]

fig2 = px.scatter(df_duration, x='start_year', y='years_running', 
                  color='status', 
                  hover_name='title',
                  # AGGIUNTO: passiamo i dati extra per il tooltip
                  hover_data={'end_date_str': True, 'start_year': True, 'years_running': ':.1f'}, 
                  title='2. Sopravvivenza - Durata in Anni per Data di Debutto',
                  labels={'start_year': 'Anno di Debutto', 'years_running': 'Anni', 'status': 'Stato', 'end_date_str': 'Fine'},
                  color_discrete_map={'Currently Airing': '#00CC96', 'Finished Airing': '#EF553B'}, 
                  opacity=0.7,
                  template='plotly_dark')

fig2.update_layout(
    xaxis_title="Anno di Inizio", 
    yaxis_title="Durata Totale (Anni)",
    legend_title_text="Stato Attuale"
)
fig2.show()

In [ ]:
# --- Grafico 3 ---

# 1. Preparazione Dati
df_rating_evo = df.dropna(subset=['score', 'start_year']).copy()
df_rating_evo['start_year'] = df_rating_evo['start_year'].astype(int)
df_rating_evo['period'] = (df_rating_evo['start_year'] // 3 * 3) 
df_rating_evo['score_round'] = df_rating_evo['score'].round().astype(int)

# 2. Creazione Griglia Completa (Stabilità)
all_periods = sorted(df_rating_evo['period'].unique())
all_scores = list(range(1, 11))
idx = pd.MultiIndex.from_product([all_periods, all_scores], names=['period', 'score_round'])

raw_counts = df_rating_evo.groupby(['period', 'score_round']).size()
full_counts = raw_counts.reindex(idx, fill_value=0).reset_index(name='count')
full_counts['period_label'] = full_counts['period'].astype(str) + "-" + (full_counts['period'] + 2).astype(str)

# Calcolo Percentuali
period_totals = full_counts.groupby('period')['count'].transform('sum')
full_counts['percent'] = full_counts.apply(
    lambda x: x['count'] / period_totals[x.name] if period_totals[x.name] > 0 else 0, axis=1
)

# 3. Grafico Animato
fig_rating_anim = px.bar(
    full_counts,
    x='score_round',
    y='percent',
    animation_frame='period_label', 
    
    title='La Qualità nel Tempo: Le Curve di Distribuzione',
    labels={'score_round': 'Rating (Voto)', 'percent': 'Distribuzione', 'period_label': 'Triennio'},
    
    range_x=[0.5, 10.5], 
    range_y=[0, 0.6],    
    
    color='score_round', 
    range_color=[1, 10], 
    color_continuous_scale='RdYlGn',
    template='plotly_dark'
)

fig_rating_anim.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1), 
    yaxis_tickformat='.0%',
    yaxis_title="Percentuale sul Totale del Periodo",
    showlegend=False,
    coloraxis_showscale=False 
)

# VELOCITÀ AGGIORNATA: 750ms (25% più veloce rispetto ai 1000ms precedenti)
fig_rating_anim.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 750
fig_rating_anim.show()

### 📊 La Chimica dell'Immortalità: Di cosa sono fatti gli anime eterni?

Il grafico animato qui sotto ci permette di osservare la Selezione Naturale in tempo reale. Premendo Play, viaggeremo attraverso le fasce di longevità:

- Start (Stagionali): Il caos primordiale. Migliaia di titoli, dominati spesso dalla Comedy e dalle storie leggere nate per consumarsi in una stagione.
- Evoluzione: Man mano che gli anni passano, i numeri crollano (notate come cambia la scala dell'asse Y!).
- End (Eterne): Chi resta in piedi dopo 10 anni? La composizione cambia drasticamente. La commedia pura lascia spesso spazio all'Avventura e all'Azione shonen, i motori narrativi infiniti per eccellenza.

In [169]:
# Gradico 4

df_comp = df.explode('genres_list').copy()

start = pd.to_datetime(df_comp['start_date'], utc=True, errors='coerce')
end = pd.to_datetime(df_comp['end_date'].fillna(pd.Timestamp.now()), utc=True, errors='coerce')
df_comp['years_running'] = (end - start).dt.days / 365.25

bins = [0, 1, 2, 5, 10, float('inf')]
labels_dur = ['Stagionale (<1)', 'Breve (1-2)', 'Media (2-5)', 'Lunga (5-10)', 'Eterna (>10)']
df_comp['life_class'] = pd.cut(df_comp['years_running'], bins=bins, labels=labels_dur, right=False)

top_genres = df_comp['genres_list'].value_counts().head(10).index
df_comp = df_comp[df_comp['genres_list'].isin(top_genres)]
df_comp = df_comp.dropna(subset=['life_class'])

comp_counts = df_comp.groupby(['life_class', 'genres_list'], observed=False).size().reset_index(name='count')
totals = comp_counts.groupby('life_class', observed=False)['count'].transform('sum')
comp_counts['percent'] = comp_counts['count'] / totals

y_max = comp_counts['percent'].max() * 1.15

fig_anim = px.bar(
    comp_counts, 
    x='genres_list', 
    y='percent', 
    color='genres_list',
    animation_frame='life_class', 
    
    text_auto='.1%',
    
    title='Grafico 4: Come cambia il DNA degli Anime',
    labels={'life_class': 'Longevità', 'percent': 'Quota di Mercato', 'genres_list': 'Genere'},
    
    category_orders={
        'life_class': labels_dur,
        'genres_list': list(top_genres)
    },
    
    range_y=[0, y_max],
    
    color_discrete_sequence=px.colors.qualitative.Pastel, 
    template='plotly_dark'
)

fig_anim.update_layout(
    yaxis_tickformat='.0%',
    xaxis_title=None, 
    yaxis_title="Composizione del Mix",
    showlegend=False,
    margin=dict(t=80, b=40)
)

fig_anim.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 1500

fig_anim.show()

## Capitolo 2: I Segreti della Sopravvivenza 🧬

La longevità non è solo una questione di quanti episodi riesci a produrre, ma di quanto a lungo riesci a restare rilevante. Alcuni anime bruciano intensamente in 24 episodi ma scompaiono in sei mesi; altri, magari con pause e stagioni distanziate, rimangono in vita per decenni.

In questo capitolo indaghiamo il 'DNA' della sopravvivenza. Esamineremo tre fattori critici:

- **La Fonte**: Essere un Manga offre una garanzia di vita più lunga rispetto a un'opera Originale?
- **Il Genere**: Ci sono generi che strutturalmente si prestano meglio a durare negli anni?
- **Lo Studio**: Esistono case di produzione specializzate nel mantenere in vita i franchise?



### 🔍 Analisi 1: La Fonte (Il Combustibile)

In [172]:
# --- 1. Preparazione Dati per Grafico 5 (Strip Plot Ibrido) ---
df_clean_duration = df.copy()

# Calcolo Durata in Anni
start = pd.to_datetime(df_clean_duration['start_date'], utc=True, errors='coerce')
end = pd.to_datetime(df_clean_duration['end_date'].fillna(pd.Timestamp.now()), utc=True, errors='coerce')
df_clean_duration['duration_years'] = (end - start).dt.days / 365.25

# Pulizia Base
df_clean_duration = df_clean_duration.dropna(subset=['duration_years', 'source'])
df_clean_duration = df_clean_duration[df_clean_duration['duration_years'] > 0]

# --- FILTRO AGGIUNTIVO: Rimozione Unknown ---
# Rimuoviamo qualsiasi fonte che contenga "Unknown"
df_clean_duration = df_clean_duration[df_clean_duration['source'] != 'Unknown']

# Filtro Top Sources (Ricalcolato DOPO aver tolto Unknown)
top_sources = df_clean_duration['source'].value_counts().head(10).index
df_clean_duration = df_clean_duration[df_clean_duration['source'].isin(top_sources)]

# --- 2. Creazione Grafico Ibrido ---
fig_source_time = px.strip(df_clean_duration, 
                           x='duration_years', 
                           y='source',
                           color='source',
                           orientation='h',
                           stripmode='overlay', 
                           title='Grafico 5: Distribuzione della Durata per Fonte (No Unknown)',
                           labels={'source': '', 'duration_years': 'Anni di Messa in Onda'},
                           template='plotly_dark',
                           hover_data=['title'])

# Aggiunta Box Plot (Nascondo legenda duplicata)
box_trace = px.box(df_clean_duration, x='duration_years', y='source', orientation='h', color='source').data[0]
box_trace.showlegend = False 
box_trace.marker.color = 'rgba(0,0,0,0)' 
box_trace.fillcolor = 'rgba(255,255,255,0.1)' 

fig_source_time.add_trace(box_trace)

# Estetica
fig_source_time.update_traces(marker=dict(size=4, opacity=0.6), selector=dict(mode='markers')) 

# Ordiniamo l'asse Y per mediana (dal più longevo al meno longevo)
fig_source_time.update_layout(yaxis={'categoryorder':'median ascending'})

fig_source_time.show()

### 🎭 L'Architettura del Genere: Chi invecchia meglio?

In [125]:
# Grafico 6

df_genres_expl = df_clean_duration.explode('genres_list')

genre_longevity = df_genres_expl.groupby('genres_list')['duration_years'].agg(['mean', 'count']).reset_index()

genre_longevity = genre_longevity[genre_longevity['count'] > 50]

top_long_genres = genre_longevity.sort_values('mean', ascending=True).tail(15)

fig_genre_time = px.bar(top_long_genres, x='mean', y='genres_list', orientation='h',
                        title='Grafico 6: Durata media in anni per Genere',
                        labels={'mean': 'Durata Media (Anni)', 'genres_list': 'Genere'},
                        color='mean',
                        color_continuous_scale='Viridis',
                        template='plotly_dark',
                        text_auto='.1f')

fig_genre_time.update_layout(xaxis_title="Anni Medi di Attività", yaxis_title="")
fig_genre_time.show()

### 🏭 3. I Costruttori di Piramidi: L'Impatto dello Studio

In [126]:
# Grafico 7

df_studios_expl = df_clean_duration.explode('studios_list')

studio_stats = df_studios_expl.groupby('studios_list').agg(
    avg_durationpy=('duration_years', 'mean'),
    max_duration=('duration_years', 'max'),
    anime_count=('anime_id', 'count')
).reset_index()

top_studios = studio_stats[studio_stats['anime_count'] > 20]

fig_studio_time = px.scatter(top_studios, x='anime_count', y='avg_durationpy',
                             size='max_duration',
                             color='avg_durationpy',
                             hover_name='studios_list',
                             title='Grafico 7: Studi per Produzione e Longevità Media',
                             labels={'anime_count': 'Numero Anime Prodotti', 
                                     'avg_durationpy': 'Longevità Media delle Serie (Anni)',
                                     'max_duration': 'Massima Durata Record (Anni)'},
                             template='plotly_dark',
                             color_continuous_scale='Plasma')

fig_studio_time.update_layout(xaxis_type="log",
                              xaxis_title="Numero Anime Prodotti (Log)",
                              yaxis_title="Durata Media dei Progetti (Anni)")
fig_studio_time.show()

## Capitolo 3: Il "Club dei Decennali": Qualità o Inerzia? 🏺

Esiste una soglia psicologica e produttiva fondamentale: i **10 anni**.
Superare un decennio di messa in onda significa aver accompagnato una generazione dall'infanzia all'età adulta. Ma cosa succede alla qualità in questo lasso di tempo?

C'è il sospetto che alcuni anime sopravvivano per pura "inerzia cult": nessuno ha il coraggio di cancellarli, e i fan continuano a guardarli per abitudine ("Sindrome di Stoccolma"), anche se la qualità narrativa è crollata.

In questa sezione isoliamo solo i **Titani (>10 anni di attività)**, sia terminati che ancora in corso, e li mettiamo sotto la lente d'ingrandimento.
1.  **Il confronto crudele:** Un anime decennale ha davvero voti migliori della media o è solo nostalgia?
2.  **La mappa degli Immortali:** Chi sono questi sopravvissuti e dove si posizionano tra capolavori e opere dimenticabili?
3.  **La Top List:** Vediamo in faccia i nomi dei più longevi e giudichiamo il loro punteggio.

In [127]:
# Grafico 8

df_clean_duration['category'] = df_clean_duration['duration_years'].apply(lambda x: 'Legendary (>10 Anni)' if x >= 10 else 'Standard (<10 Anni)')

fig_compare = px.box(df_clean_duration, 
                     x='category', 
                     y='score', 
                     color='category',
                     title='Grafico 8: I "Decennali" sono davvero migliori?',
                     labels={'category': 'Categoria', 'score': 'Voto Medio (Score)'},
                     color_discrete_map={'Legendary (>10 Anni)': '#FFA15A', 'Standard (<10 Anni)': '#636EFA'},
                     template='plotly_dark',
                     points=False)

fig_compare.update_layout(showlegend=False, xaxis_title="")
fig_compare.show()

In [128]:
# Grafico 9

df_titans = df.copy()

start = pd.to_datetime(df_titans['start_date'], utc=True, errors='coerce')
end = pd.to_datetime(df_titans['end_date'].fillna(pd.Timestamp.now()), utc=True, errors='coerce')
df_titans['duration_years'] = (end - start).dt.days / 365.25

MIN_DURATION = 5  

df_titans = df_titans[
    (df_titans['duration_years'] >= MIN_DURATION) & 
    (df_titans['score'] > 0)
].copy()

df_titans['log_members'] = np.log10(df_titans['members'] + 1)

fig_titans_clean = px.scatter(
    df_titans, 
    x='duration_years', 
    y='score', 
    size='log_members',      
    color='status',          
    hover_name='title',
    
    title='Grafico 9: I Sopravvissuti (>5 Anni)',
    labels={'duration_years': 'Longevità (Anni)', 'score': 'Rating', 'status': 'Stato'},
    
    color_discrete_map={'Currently Airing': '#00ddff', 'Finished Airing': '#ff5555'}, 
    template='plotly_dark'
)

med_dur = df_titans['duration_years'].median()
med_score = df_titans['score'].median()

fig_titans_clean.add_vline(x=med_dur, line_dash="dash", line_color="grey")
fig_titans_clean.add_hline(y=med_score, line_dash="dash", line_color="grey")

fig_titans_clean.add_annotation(x=df_titans['duration_years'].max(), y=9.5, text="Eterni & Leggendari", 
                                showarrow=False, font=dict(color="gold", size=12), xanchor="right", yanchor="top")
fig_titans_clean.add_annotation(x=MIN_DURATION, y=9.5, text="Intensi & Amati", 
                                showarrow=False, font=dict(color="cyan", size=12), xanchor="left", yanchor="top")
fig_titans_clean.add_annotation(x=df_titans['duration_years'].max(), y=5, text="Lunga vita, bassa qualità", 
                                showarrow=False, font=dict(color="orange", size=12), xanchor="right", yanchor="bottom")

fig_titans_clean.update_traces(
    marker=dict(opacity=0.8, line=dict(width=1, color='Black')) 
)

fig_titans_clean.update_layout(
    height=750,
    xaxis_title="Anni di Messa in Onda",
    yaxis_title="Rating Medio",
    margin=dict(t=50, l=50, r=50, b=50)
)

fig_titans_clean.show()

In [129]:
# Grafico 10: Bar Chart dei Top 20
top_20_longest = df_titans.sort_values(by='duration_years', ascending=False).head(20)

fig_top20 = px.bar(top_20_longest, 
                   y='title', 
                   x='duration_years', 
                   color='score',
                   orientation='h',
                   title='Grafico 10: I 20 Anime più longevi della storia',
                   labels={'title': 'Anime', 'duration_years': 'Anni', 'score': 'Voto'},
                   color_continuous_scale='RdYlGn', 
                   text='duration_years',
                   template='plotly_dark')

fig_top20.update_traces(texttemplate='%{text:.1f} anni', textposition='outside')
fig_top20.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_title="Anni di Trasmissione",
    yaxis_title="",
    height=800
)
fig_top20.show()

## Capitolo 4: Il Cimitero degli Spettatori 💀

Ogni anno in più che una serie rimane in onda è una scommessa sulla pazienza del pubblico.
Il **Drop Rate** (percentuale di utenti che abbandonano la serie) è la misura più onesta della "fatica" narrativa.

Analizziamo il rapporto tra tempo e abbandono su 3 livelli:
1.  **Il Peso del Tempo:** I progetti a lungo termine spaventano il pubblico?
2.  **Le Fasce di Rischio:** Esiste una durata "maledetta" in cui tutti mollano?
3.  **Il Muro dei Titani:** Quali sono i giganti che hanno perso più spettatori lungo la strada?

### 1. Il Paradosso dell'Impegno 📉
Si potrebbe pensare che una serie lunghissima abbia un drop rate più alto per pura stanchezza.
Tuttavia, esiste il fenomeno opposto: il **"Filtro del Fan"**. Chi inizia una serie di 10 anni sa già a cosa va incontro. Spesso sono le serie "medie" (né brevi né infinite) a soffrire di crisi d'identità e perdere pubblico.

Il grafico seguente cerca una correlazione tra Anni di Attività e Percentuale di Abbandono.

In [162]:
# Grafico 11 Scatter: Durata vs Drop Rate
df_clean_duration['drop_rate'] = (df_clean_duration['dropped'] / df_clean_duration['total']) * 100

df_drop = df_clean_duration[df_clean_duration['total'] > 500].copy()

fig_drop_trend = px.scatter(df_drop, 
                            x='duration_years', 
                            y='drop_rate', 
                            opacity=0.4,
                            trendline="lowess", # Usa una trendline curva per vedere l'andamento non lineare
                            trendline_color_override="cyan",
                            title='Grafico 11: Più dura, più viene abbandonato?',
                            labels={'duration_years': 'Anni di Attività', 'drop_rate': 'Tasso di Abbandono (%)'},
                            hover_data=['title'],
                            template='plotly_dark')

fig_drop_trend.update_layout(
    xaxis_title="Anni di Messa in Onda",
    yaxis_title="Percentuale Utenti che hanno 'Droppato'",
    yaxis_range=[0, 30]
)
fig_drop_trend.show()

### 2. La Zona di Pericolo ⚠️
Non tutte le durate sono uguali.
*   **Stagionali (<1 anno):** Basso rischio, basso impegno.
*   **Medie (1-5 anni):** Il limbo. Abbastanza lunghe da stancare, non abbastanza da diventare icone.
*   **Infinite (>10 anni):** Richiedono fede assoluta.

Il Box Plot qui sotto svela la verità: sorprendentemente, le serie **"Eterne"** hanno spesso tassi di abbandono più bassi (mediana inferiore) rispetto alle serie medie. Perché? Perché chi guarda *One Piece* o *Doraemon* fa parte di una "setta" fedele. Sono le serie di 2-5 anni quelle che rischiano di più di perdere interesse per strada.

In [137]:
# 12. Violin Plot per Fasce
bins = [0, 1, 5, 10, 100]
labels = ['Stagionali <1', 'Brevi 1-5', 'Medie 5-10', 'Immortal > 10 ']
df_drop['risk_zone'] = pd.cut(df_drop['duration_years'], bins=bins, labels=labels)

fig_risk = px.violin(df_drop, 
                     x='risk_zone', 
                     y='drop_rate', 
                     color='risk_zone',
                     box=True,
                     points=False,
                     title='Grafico 12: Distribuzione Drop Rate per durata',
                     labels={'risk_zone': 'Tipo di Durata', 'drop_rate': 'Drop Rate (%)'},
                     template='plotly_dark')

fig_risk.update_layout(showlegend=False, xaxis_title="", yaxis_title="Drop Rate (%)")
fig_risk.show()

### 3. Il Muro del Pianto: I Titani più abbandonati 🧱
Essere famosi e longevi non salva dall'abbandono. Alcuni anime iconici hanno un "muro d'ingresso" altissimo.
Può essere dovuto a filler infiniti, cali di qualità o semplicemente una trama che si è persa.

In questo grafico isoliamo solo i **Titani (>10 anni)** e mostriamo la **Top 10 per Tasso di Abbandono**.
Questi sono i giganti che, nonostante la loro fama, hanno lasciato più "caduti" sul campo di battaglia.

In [163]:
# 13. Bar Chart
titans_drop = df_drop[df_drop['duration_years'] >= 5].copy()

titans_drop_major = titans_drop[titans_drop['total'] > 10000].sort_values(by='drop_rate', ascending=False).head(10)

fig_wall = px.bar(titans_drop_major, 
                  y='title', 
                  x='drop_rate', 
                  color='drop_rate',
                  orientation='h',
                  title='Gradico 13: Top 10 Titani per Tasso di Abbandono più alto',
                  labels={'title': '', 'drop_rate': 'Drop Rate (%)'},
                  color_continuous_scale='Reds',
                  text='duration_years',
                  template='plotly_dark')

fig_wall.update_traces(texttemplate='%{x:.1f}% (da %{text:.0f} anni)', textposition='inside')
fig_wall.update_layout(yaxis={'categoryorder':'total ascending'}, xaxis_title="Percentuale Utenti che hanno abbandonato")
fig_wall.show()

# Capitolo 5: I Titani del Presente

In [164]:
# 1. DEFINIAMO df_ongoing
# Filtriamo solo le serie attualmente in corso
df_ongoing = df[df['status'] == 'Currently Airing'].copy()

# Usiamo tz='UTC' per renderlo compatibile con le date del dataset
current_date = pd.Timestamp.now(tz='UTC')
df_ongoing['current_duration_years'] = (current_date - df_ongoing['start_date']).dt.days / 365.25

# Pulizia dati
df_ongoing = df_ongoing[df_ongoing['current_duration_years'] > 0]

# Creiamo le categorie
df_ongoing['category'] = list(map(lambda x: 'Veterani (>10 anni)' if x >= 10 else 'Nuove Leve (<10 anni)', df_ongoing['current_duration_years']))


### 1. L'Abisso Temporale: I Pochissimi Eletti
Osservando il panorama degli anime attualmente in onda, notiamo una frattura netta. La quasi totalità delle produzioni odierne ha meno di 5 anni di vita.
Esiste però un ristretto club di **"Immortali"**: serie iniziate negli anni '60, '70 o '90 che continuano imperterrite.
Il grafico sottostante mostra questo "abisso": una nuvola densissima di nuove uscite (in basso a destra) e pochissimi punti isolati che si estendono verso l'alto a sinistra (Sazae-san, Doraemon, One Piece). Non c'è una distribuzione uniforme: o sei una hit del momento, o sei un'istituzione decennale.

In [134]:
# 14.
df_plot = df_ongoing.dropna(subset=['score']).copy()

fig_ongoing_better = px.scatter(df_plot, 
                                x='current_duration_years', 
                                y='members',
                                color='category', 
                                size='score',
                                hover_name='title',
                                title='Grafico 14: I Veterani sono tutti Famosi?',
                                labels={'current_duration_years': 'Anni di Trasmissione', 'members': 'Popolarità (Members)', 'score': 'Voto'},
                                color_discrete_map={'Veterani (>10 anni)': '#EF553B', 'Nuove Leve (<10 anni)': '#636efa'},
                                log_y=True,
                                template='plotly_dark')

fig_ongoing_better.update_layout(height=600)
fig_ongoing_better.show()

### 2. I Veri Re dello Schermo

Abbiamo parlato di sopravvivenza, ma chi sono i reali dominatori del tempo? Il grafico seguente isola le 15 serie ancora in corso più longeve in assoluto col miglior rating.

Qui la prospettiva cambia:

- La Lunghezza delle Barre indica la Qualità (il voto medio degli utenti).
- Il Colore indica l'Anzianità (dal giallo dei "giovani" al rosso cupo degli "antichi").

Siamo di fronte a capolavori intramontabili o a prodotti che sopravvivono per inerzia? Notate come alcune serie "giovani" (in giallo) abbiano già superato in gradimento colossi storici (in rosso scuro), suggerendo che l'anzianità non sempre garantisce l'eccellenza.

In [135]:
# 15.
df_ongoing = df[df['status'] == 'Currently Airing'].copy()
start = pd.to_datetime(df_ongoing['start_date'], utc=True, errors='coerce')
end = pd.Timestamp.now(tz='UTC')
df_ongoing['current_duration_years'] = (end - start).dt.days / 365.25

df_ongoing = df_ongoing.dropna(subset=['score'])
top_15_ongoing = df_ongoing.sort_values(by='current_duration_years', ascending=False).head(15)

fig_top_ongoing = px.bar(
    top_15_ongoing, 
    x='score', 
    y='title', 
    color='current_duration_years', 
    orientation='h',
    
    title='Grafico 15: Le serie eterne',
    labels={
        'title': '', 
        'current_duration_years': 'Anni in onda', 
        'score': 'Rating (1-10)'
    },

    color_continuous_scale='YlOrRd', 
    
    text='score', 
    template='plotly_dark'
)

fig_top_ongoing.update_layout(
    yaxis={'categoryorder':'total ascending'},
    height=700,
    coloraxis_colorbar_title="Anni di<br>Vita",
    xaxis_title="Rating Utenti"
)

fig_top_ongoing.update_traces(
    texttemplate='%{text:.2f} ⭐', 
    textposition='inside',
    textfont=dict(color='black') 
)

min_score = top_15_ongoing['score'].min() * 0.9
fig_top_ongoing.update_xaxes(range=[min_score, 10]) 

fig_top_ongoing.show()

## 🏁 Conclusioni: Il Segreto dell'Immortalità

In [136]:
# --- GRAFICO FINALE: "LA GALASSIA DELL'ANIMAZIONE" ---

df_final = df.copy()

calc_end_date = pd.to_datetime(df_final['end_date'].fillna(pd.Timestamp.now(tz='UTC')), utc=True)
start_date_final = pd.to_datetime(df_final['start_date'], errors='coerce', utc=True)
df_final['final_duration'] = (calc_end_date - start_date_final).dt.days / 365.25

df_final = df_final.dropna(subset=['score', 'start_year', 'final_duration'])
df_final = df_final[df_final['final_duration'] > 0]
df_final = df_final[df_final['start_year'] >= 1960]

fig_galaxy = px.scatter(df_final, 
                        x='start_year', 
                        y='score', 
                        size='final_duration',
                        color='status',
                        hover_name='title',
                        hover_data={'final_duration': ':.1f', 'members': True},
                        title='<b>La Galassia dell\'Animazione</b>: Storia, Qualità e Sopravvivenza',
                        labels={'start_year': 'Anno di Nascita', 'score': 'Qualità (Score)', 'final_duration': 'Longevità (Anni)'},
                        color_discrete_map={'Currently Airing': '#00CC96', 'Finished Airing': '#EF553B', 'Not yet aired': 'gray'},
                        size_max=40,
                        opacity=0.6,
                        template='plotly_dark')

fig_galaxy.update_layout(
    height=800,
    xaxis_title="Linea Temporale",
    yaxis_title="Rating (Qualità)",
    legend_title_text="Stato Vitale"
)

fig_galaxy.show()

Siamo partiti con una domanda provocatoria: *gli anime longevi sono opere d'arte che resistono al tempo o prodotti commerciali che sopravvivono per inerzia?*
I dati hanno parlato chiaro, smentendo alcuni pregiudizi e confermando nuove verità:

### 1. Il Darwinismo è Reale (Premessa Smentita)
L'idea che le serie lunghe "trascinino" stancamente i voti per abitudine è **falsa**. I dati mostrano una correlazione positiva tra longevità e punteggio.
Il mercato dell'anime è spietato: non c'è spazio per mediocrità duratura. Le opere che superano il decennio (i "Titani") sono quasi esclusivamente eccellenze (Score > 7.5). Se la qualità cala, la serie viene cancellata molto prima di diventare "immortale". La longevità non è un caso, è un **certificato di qualità**.

### 2. I Veri Immortali sono Invisibili
Abbiamo scoperto che i veri dominatori del tempo non sono le hit globali come *One Piece* o *Naruto*, ma le serie per famiglie (*Sazae-san*, *Doraemon*).
Questi anime operano su un livello diverso: non puntano all'hype o al trend del momento, ma all'indispensabilità sociale. Sono diventati riti quotidiani, immuni alle mode.

### 3. L'Esplosione Effimera
L'analisi storica mostra un'esplosione produttiva negli ultimi dieci anni (la "Bolla dello Streaming"), ma accompagnata da una frammentazione della durata. Si produce di più, ma si consuma (e si dimentica) più in fretta.
La "Galassia" qui sotto lo dimostra visivamente: una densità enorme di piccole stelle recenti di alta qualità ma vita breve, contro poche, giganti stelle antiche che continuano a bruciare.